# RSNA Knee — Phase 4B-2B1
## Continue training → locked gold evaluation → visual MRI/report QA

This single notebook continues from the already-completed **selection stage**.

Frozen result from the previous run:

```text
selection fold: 0
selection train: 3437
selection pseudo-val: 904
gold locked: 58

best epoch: 5
best pseudo-val macro weighted BCE ≈ 0.45337
epoch 6 worsened ≈ 0.46362
```

Therefore model selection is complete.

This notebook does:

```text
selection-best checkpoint
        ↓
clean final refit on all 4,341 PSEUDO_TRAIN studies
        ↓
exactly 5 epochs
        ↓
save final model
        ↓
locked 58-study gold evaluation
        ↓
results visualizations
        ↓
MRI + attention + clinical-report case review
```

## Important scientific rule

The clinical report and the 58 locked gold labels are **not** used to train or select this MRI model.

The report section at the end is only a post-hoc diagnostic to help us understand how:

```text
MRI model
vs
frozen report-derived teacher
vs
official gold label
```

behave on the same study.

This is **not yet a multimodal fused submission model**.

In [ ]:
from pathlib import Path
from functools import lru_cache
from hashlib import md5
import json, os, re, time, math, random, warnings, zipfile, shutil

import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models import resnet18, ResNet18_Weights

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

TARGET_SLUG = {
    "ACL": "acl",
    "MCL": "mcl",
    "Medial Meniscus": "medial_meniscus",
    "Lateral Meniscus": "lateral_meniscus",
    "Medial OA": "medial_oa",
    "Lateral OA": "lateral_oa",
    "PF OA": "pf_oa",
    "Effusion": "effusion",
    "Synovitis": "synovitis",
    "Baker's": "bakers",
    "Contusion": "contusion",
    "Fracture": "fracture",
}

SLOTS = [
    "SAG_FS", "SAG_NONFS",
    "COR_FS", "COR_NONFS",
    "AX_FS", "AX_NONFS",
]
SLOT_TO_ID = {s: i for i, s in enumerate(SLOTS)}

CROP_MM = 130.0
IMAGE_SIZE = 224
N_ANCHORS = 8
SLICE_BAND = (0.12, 0.88)

COMP_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


EXPERIMENT_VERSION = "PHASE4B2B1_SIXSLOT_TARGETATTN_RESNET18_v1"

BATCH_SIZE = 2
NUM_WORKERS = 2

BEST_EPOCH = 5
BACKBONE_LR = 3e-5
HEAD_LR = 2e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.20
GRAD_CLIP = 2.0

In [ ]:
SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("/mnt/data"),
    Path("."),
]

def all_named(filename):
    hits = []
    for root in SEARCH_ROOTS:
        if root.exists():
            try:
                hits.extend(root.rglob(filename))
            except Exception:
                pass

    out, seen = [], set()

    for p in hits:
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)

    return out


def resolve_one(filename, required=True):
    hits = all_named(filename)

    if len(hits) == 1:
        return hits[0]

    kaggle_hits = [
        p for p in hits
        if str(p).startswith("/kaggle/input/")
    ]

    if kaggle_hits:
        chosen = sorted(
            kaggle_hits,
            key=lambda p: (len(str(p)), str(p))
        )[0]

        if len(hits) > 1:
            print(f"Multiple {filename}; using {chosen}")

        return chosen

    if hits:
        return sorted(hits, key=lambda p: str(p))[0]

    if required:
        raise FileNotFoundError(filename)

    return None


SLOT_PLAN_PATH = resolve_one(
    "rsna_phase4b2a_six_slot_plan_v1.csv"
)

PSEUDO_WIDE_PATH = resolve_one(
    "rsna_pseudo_labels_v1_1_wide.csv"
)

PSEUDO_JSON_PATH = resolve_one(
    "PSEUDO_LABELS_v1_1.json"
)

TRAIN_CSV = COMP_DIR / "train.csv"

assert TRAIN_CSV.exists(), TRAIN_CSV

print("Six-slot plan:", SLOT_PLAN_PATH)
print("Pseudo labels:", PSEUDO_WIDE_PATH)
print("Pseudo freeze:", PSEUDO_JSON_PATH)

## 1. Restore and verify the selection checkpoint

In [ ]:
USER_SELECTION_PATH = Path(
    "/kaggle/input/models/aymenslimani/"
    "phase4b2b1-selection-best-v1-/other/default/1/"
    "phase4b2b1_selection_best_v1"
)

def looks_like_extracted_torch_archive(path):
    return (
        path.is_dir()
        and (path / "data.pkl").exists()
        and (path / "data").is_dir()
        and (path / "version").exists()
    )


def repack_extracted_torch_archive(src_dir, out_file):
    src_dir = Path(src_dir)
    out_file = Path(out_file)

    prefix = src_dir.name or "checkpoint"

    with zipfile.ZipFile(
        out_file,
        "w",
        compression=zipfile.ZIP_STORED,
    ) as zf:
        for f in src_dir.rglob("*"):
            if not f.is_file():
                continue

            rel = f.relative_to(src_dir)
            arcname = f"{prefix}/{rel.as_posix()}"

            zf.write(
                f,
                arcname=arcname,
            )

    return out_file


def discover_selection_checkpoint():
    # Exact user path first.
    if USER_SELECTION_PATH.exists():
        return USER_SELECTION_PATH

    # Search both files and directories.
    candidates = []

    for root in [
        Path("/kaggle/input/models"),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]:
        if not root.exists():
            continue

        try:
            candidates.extend(
                root.rglob(
                    "*phase4b2b1_selection_best_v1*"
                )
            )
        except Exception:
            pass

    candidates = [
        p for p in candidates
        if (
            p.is_file()
            or looks_like_extracted_torch_archive(p)
        )
    ]

    if not candidates:
        raise FileNotFoundError(
            "Could not find the Phase 4B-2B1 selection-best checkpoint."
        )

    candidates = sorted(
        candidates,
        key=lambda p: (
            0 if looks_like_extracted_torch_archive(p) else 1,
            len(str(p)),
            str(p),
        ),
    )

    return candidates[0]


def materialize_selection_checkpoint():
    source = discover_selection_checkpoint()

    print("Selection artifact source:", source)

    if source.is_file():
        return source

    if looks_like_extracted_torch_archive(source):
        out = OUT_DIR / "phase4b2b1_selection_best_v1_repacked.pt"

        print(
            "Kaggle mounted the PyTorch checkpoint as an extracted archive."
        )
        print("Repacking it to:", out)

        return repack_extracted_torch_archive(
            source,
            out,
        )

    # Maybe the selected path is a parent directory.
    archives = [
        p for p in source.rglob("*")
        if looks_like_extracted_torch_archive(p)
    ]

    if archives:
        out = OUT_DIR / "phase4b2b1_selection_best_v1_repacked.pt"

        return repack_extracted_torch_archive(
            archives[0],
            out,
        )

    files = [
        p for p in source.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".pt", ".pth", ".ckpt"}
    ]

    if files:
        return files[0]

    raise RuntimeError(
        f"Selection artifact exists but could not be materialized: {source}"
    )


SELECTION_CHECKPOINT_PATH = materialize_selection_checkpoint()

selection_checkpoint = torch.load(
    SELECTION_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print()
print("Checkpoint keys:", list(selection_checkpoint.keys()))
print("Experiment:", selection_checkpoint.get("experiment_version"))
print("Best epoch:", selection_checkpoint.get("best_epoch"))
print("Best pseudo-val loss:", selection_checkpoint.get("best_pseudo_val_loss"))

assert int(selection_checkpoint["best_epoch"]) == 5

print("\n✅ Selection checkpoint restored")

In [ ]:
assert int(
    selection_checkpoint["best_epoch"]
) == BEST_EPOCH

print(
    "Frozen refit epoch count:",
    BEST_EPOCH,
)

print(
    "Selection pseudo-val:",
    selection_checkpoint[
        "best_pseudo_val_loss"
    ],
)

# Free selection model state from CPU memory after metadata verification.
selection_checkpoint = {
    "experiment_version":
        selection_checkpoint.get(
            "experiment_version"
        ),

    "best_epoch":
        int(
            selection_checkpoint[
                "best_epoch"
            ]
        ),

    "best_pseudo_val_loss":
        float(
            selection_checkpoint[
                "best_pseudo_val_loss"
            ]
        ),
}

## 2. Load supervision and define the DICOM pipeline

In [ ]:
slot_plan = pd.read_csv(SLOT_PLAN_PATH)
pseudo = pd.read_csv(PSEUDO_WIDE_PATH)
train_csv = pd.read_csv(TRAIN_CSV)

slot_plan["StudyInstanceUID"] = slot_plan["StudyInstanceUID"].astype(str)
slot_plan["SeriesInstanceUID"] = slot_plan["SeriesInstanceUID"].astype(str)
pseudo["StudyInstanceUID"] = pseudo["StudyInstanceUID"].astype(str)
train_csv["StudyInstanceUID"] = train_csv["StudyInstanceUID"].astype(str)

PSEUDO_COLS = [
    f"pseudo_{TARGET_SLUG[t]}"
    for t in TARGETS
]

WEIGHT_COLS = [
    f"weight_{TARGET_SLUG[t]}"
    for t in TARGETS
]

pseudo_indexed = pseudo.set_index("StudyInstanceUID")

slot_rows_by_uid = {
    uid: {
        r.slot: r
        for r in d.itertuples(index=False)
    }
    for uid, d in slot_plan.groupby("StudyInstanceUID")
}


def _header_order_record(path):
    try:
        ds = pydicom.dcmread(
            str(path),
            stop_before_pixels=True,
            force=True,
            specific_tags=[
                "ImagePositionPatient",
                "ImageOrientationPatient",
                "InstanceNumber",
            ],
        )

        ipp = getattr(ds, "ImagePositionPatient", None)
        iop = getattr(ds, "ImageOrientationPatient", None)
        inst = getattr(ds, "InstanceNumber", None)

        spatial = None

        if (
            ipp is not None
            and iop is not None
            and len(ipp) >= 3
            and len(iop) >= 6
        ):
            row = np.asarray(iop[:3], dtype=np.float64)
            col = np.asarray(iop[3:6], dtype=np.float64)
            normal = np.cross(row, col)

            norm = np.linalg.norm(normal)

            if norm > 1e-8:
                normal /= norm
                spatial = float(
                    np.dot(
                        np.asarray(ipp[:3], dtype=np.float64),
                        normal,
                    )
                )

        inst_value = (
            float(inst)
            if inst is not None
            else np.nan
        )

        return str(path), spatial, inst_value

    except Exception:
        return str(path), None, np.nan


@lru_cache(maxsize=30000)
def ordered_series_files(series_path):
    folder = Path(series_path)

    files = sorted([
        p for p in folder.iterdir()
        if p.is_file()
    ])

    records = [
        _header_order_record(p)
        for p in files
    ]

    n_spatial = sum(
        r[1] is not None
        for r in records
    )

    if n_spatial >= max(2, int(0.8 * len(records))):
        records = sorted(
            records,
            key=lambda r: (
                float("inf")
                if r[1] is None
                else r[1],
                r[0],
            ),
        )

    elif (
        np.isfinite([r[2] for r in records]).sum()
        >= max(2, int(0.8 * len(records)))
    ):
        records = sorted(
            records,
            key=lambda r: (
                float("inf")
                if not np.isfinite(r[2])
                else r[2],
                r[0],
            ),
        )

    else:
        records = sorted(
            records,
            key=lambda r: r[0],
        )

    return tuple(
        r[0]
        for r in records
    )


def normalize_pixels(arr, photometric="MONOCHROME2"):
    x = np.asarray(
        arr,
        dtype=np.float32,
    )

    finite = np.isfinite(x)

    if not finite.any():
        return np.zeros_like(
            x,
            dtype=np.float32,
        )

    vals = x[finite]

    lo, hi = np.percentile(
        vals,
        [1.0, 99.0],
    )

    if hi <= lo + 1e-6:
        lo = float(vals.min())
        hi = float(vals.max())

    if hi <= lo + 1e-6:
        return np.zeros_like(
            x,
            dtype=np.float32,
        )

    x = np.clip(
        (x - lo) / (hi - lo),
        0.0,
        1.0,
    )

    if str(photometric).upper() == "MONOCHROME1":
        x = 1.0 - x

    x[~finite] = 0.0

    return x.astype(np.float32)


def center_physical_crop(
    x,
    spacing_row,
    spacing_col,
    crop_mm=CROP_MM,
):
    h, w = x.shape[:2]

    try:
        sr = float(spacing_row)
        sc = float(spacing_col)
    except Exception:
        sr = sc = np.nan

    crop_h = (
        int(round(crop_mm / sr))
        if np.isfinite(sr) and sr > 0
        else h
    )

    crop_w = (
        int(round(crop_mm / sc))
        if np.isfinite(sc) and sc > 0
        else w
    )

    crop_h = max(8, min(crop_h, h))
    crop_w = max(8, min(crop_w, w))

    y0 = max((h - crop_h) // 2, 0)
    x0 = max((w - crop_w) // 2, 0)

    return x[
        y0:y0 + crop_h,
        x0:x0 + crop_w,
    ]


def decode_one(path, spacing_row, spacing_col):
    try:
        ds = pydicom.dcmread(
            str(path),
            force=True,
        )

        arr = ds.pixel_array

        if arr.ndim > 2:
            arr = np.asarray(arr)[0]

        x = normalize_pixels(
            arr,
            getattr(
                ds,
                "PhotometricInterpretation",
                "MONOCHROME2",
            ),
        )

        x = center_physical_crop(
            x,
            spacing_row,
            spacing_col,
        )

        x = cv2.resize(
            x,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_AREA,
        )

        return x.astype(np.float32)

    except Exception:
        return None


def sample_anchor_indices(n, k=N_ANCHORS):
    if n <= 0:
        return np.zeros(k, dtype=np.int64)

    lo = int(np.floor(SLICE_BAND[0] * (n - 1)))
    hi = int(np.ceil(SLICE_BAND[1] * (n - 1)))

    if hi < lo:
        lo, hi = 0, n - 1

    return np.clip(
        np.rint(
            np.linspace(lo, hi, k)
        ),
        0,
        n - 1,
    ).astype(np.int64)


def load_25d_anchors(slot_row, k=N_ANCHORS):
    files = ordered_series_files(
        str(slot_row.series_path)
    )

    n = len(files)

    out = np.zeros(
        (k, 3, IMAGE_SIZE, IMAGE_SIZE),
        dtype=np.float32,
    )

    valid = np.zeros(
        k,
        dtype=np.float32,
    )

    if n == 0:
        return out, valid

    anchors = sample_anchor_indices(
        n,
        k,
    )

    sr = getattr(
        slot_row,
        "first_PixelSpacingRow",
        np.nan,
    )

    sc = getattr(
        slot_row,
        "first_PixelSpacingCol",
        np.nan,
    )

    for j, center_idx in enumerate(anchors):
        triplet = [
            max(center_idx - 1, 0),
            center_idx,
            min(center_idx + 1, n - 1),
        ]

        imgs = [
            decode_one(files[i], sr, sc)
            for i in triplet
        ]

        good = [
            x for x in imgs
            if x is not None
        ]

        if not good:
            continue

        fallback = good[len(good) // 2]

        imgs = [
            x if x is not None
            else fallback
            for x in imgs
        ]

        out[j] = np.stack(
            imgs,
            axis=0,
        )

        valid[j] = 1.0

    return out, valid


def build_one_study(uid):
    uid = str(uid)

    images = np.zeros(
        (
            len(SLOTS),
            N_ANCHORS,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        dtype=np.float32,
    )

    slice_mask = np.zeros(
        (len(SLOTS), N_ANCHORS),
        dtype=np.float32,
    )

    slot_mask = np.zeros(
        len(SLOTS),
        dtype=np.float32,
    )

    slot_dict = slot_rows_by_uid.get(uid, {})

    for slot_name, slot_id in SLOT_TO_ID.items():
        if slot_name not in slot_dict:
            continue

        arr, valid = load_25d_anchors(
            slot_dict[slot_name],
            N_ANCHORS,
        )

        if valid.sum() <= 0:
            continue

        images[slot_id] = arr
        slice_mask[slot_id] = valid
        slot_mask[slot_id] = 1.0

    return images, slice_mask, slot_mask

In [ ]:
with open(
    PSEUDO_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:
    pseudo_meta = json.load(f)

assert (
    pseudo_meta["freeze_name"]
    == "PSEUDO_LABELS_v1_1"
)

assert (
    pseudo_meta["n_pseudo_train_studies"]
    == 4341
)

assert (
    pseudo_meta["n_gold_eval_locked"]
    == 58
)

assert (
    pseudo_meta["n_gold_group_locked_non_gold"]
    == 6
)

assert (
    pseudo_meta[
        "gold_group_leakage_check_passed"
    ]
    is True
)

study_role = (
    slot_plan[
        [
            "StudyInstanceUID",
            "training_role",
            "cv_group_id",
        ]
    ]
    .drop_duplicates(
        "StudyInstanceUID"
    )
)

all_pseudo_uids = sorted(
    study_role.loc[
        study_role[
            "training_role"
        ].eq(
            "PSEUDO_TRAIN"
        ),
        "StudyInstanceUID",
    ]
)

gold_uids = sorted(
    study_role.loc[
        study_role[
            "training_role"
        ].eq(
            "GOLD_EVAL_LOCKED"
        ),
        "StudyInstanceUID",
    ]
)

assert len(all_pseudo_uids) == 4341
assert len(gold_uids) == 58
assert not (
    set(all_pseudo_uids)
    & set(gold_uids)
)

gold_label_table = (
    train_csv[
        [
            "StudyInstanceUID"
        ] + TARGETS
    ]
    .set_index(
        "StudyInstanceUID"
    )
)

print("Pseudo train:", len(all_pseudo_uids))
print("Gold locked:", len(gold_uids))
print("✅ Supervision freeze validated")

## 3. Study dataset

In [ ]:
class SixSlotStudyDataset(Dataset):

    def __init__(
        self,
        uids,
        training,
        use_gold_labels=False,
    ):
        self.uids = list(
            map(str, uids)
        )

        self.training = bool(training)
        self.use_gold_labels = bool(
            use_gold_labels
        )

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, idx):
        uid = self.uids[idx]

        images, slice_mask, slot_mask = (
            build_one_study(uid)
        )

        if slot_mask.sum() <= 0:
            raise RuntimeError(
                f"No decodable slot for {uid}"
            )

        if self.use_gold_labels:
            y = (
                gold_label_table.loc[
                    uid,
                    TARGETS,
                ]
                .to_numpy(
                    np.float32
                )
            )

            w = np.ones(
                len(TARGETS),
                dtype=np.float32,
            )

        else:
            row = pseudo_indexed.loc[uid]

            y = row[
                PSEUDO_COLS
            ].to_numpy(
                np.float32
            )

            w = row[
                WEIGHT_COLS
            ].to_numpy(
                np.float32
            )

        return {
            "StudyInstanceUID":
                uid,

            "images":
                torch.from_numpy(
                    images
                ),

            "slice_mask":
                torch.from_numpy(
                    slice_mask
                ),

            "slot_mask":
                torch.from_numpy(
                    slot_mask
                ),

            "targets":
                torch.from_numpy(
                    y
                ),

            "weights":
                torch.from_numpy(
                    w
                ),
        }


def make_loader(
    uids,
    training,
    use_gold_labels=False,
    batch_size=BATCH_SIZE,
):
    ds = SixSlotStudyDataset(
        uids,
        training=training,
        use_gold_labels=use_gold_labels,
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=training,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        drop_last=False,
    )

## 4. Model

In [ ]:
def load_imagenet_resnet18():
    # Official torchvision ImageNet-1K weights.
    # If already cached, no download occurs.
    m = resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    m.fc = nn.Identity()

    return m


class SixSlotTargetAttention(nn.Module):

    def __init__(
        self,
        n_targets=12,
        n_slots=6,
        embed_dim=512,
        n_heads=8,
        dropout=0.2,
    ):
        super().__init__()

        self.backbone = (
            load_imagenet_resnet18()
        )

        self.slice_score = nn.Sequential(
            nn.Linear(
                embed_dim,
                128,
            ),
            nn.Tanh(),
            nn.Linear(
                128,
                1,
            ),
        )

        self.slot_embedding = nn.Embedding(
            n_slots,
            embed_dim,
        )

        self.target_queries = nn.Parameter(
            torch.randn(
                n_targets,
                embed_dim,
            ) * 0.02
        )

        self.cross_attention = (
            nn.MultiheadAttention(
                embed_dim=embed_dim,
                num_heads=n_heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.norm1 = nn.LayerNorm(
            embed_dim
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                embed_dim,
                embed_dim * 2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                embed_dim * 2,
                embed_dim,
            ),
        )

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.target_head_weight = (
            nn.Parameter(
                torch.randn(
                    n_targets,
                    embed_dim,
                ) * 0.02
            )
        )

        self.target_head_bias = (
            nn.Parameter(
                torch.zeros(
                    n_targets
                )
            )
        )

    def forward(
        self,
        images,
        slice_mask,
        slot_mask,
    ):
        B, S, K, C, H, W = (
            images.shape
        )

        flat = images.reshape(
            B * S * K,
            C,
            H,
            W,
        )

        features = self.backbone(
            flat
        )

        features = features.reshape(
            B,
            S,
            K,
            -1,
        )

        scores = (
            self.slice_score(
                features
            ).squeeze(-1)
        )

        m = slice_mask.float()

        scores = (
            scores
            - scores.amax(
                dim=-1,
                keepdim=True,
            )
        )

        exp_scores = (
            torch.exp(scores)
            * m
        )

        attn = (
            exp_scores
            / (
                exp_scores.sum(
                    dim=-1,
                    keepdim=True,
                )
                + 1e-8
            )
        )

        slot_tokens = (
            features
            * attn.unsqueeze(-1)
        ).sum(dim=2)

        slot_ids = torch.arange(
            S,
            device=images.device,
        )

        slot_tokens = (
            slot_tokens
            + self.slot_embedding(
                slot_ids
            )[None, :, :]
        )

        queries = (
            self.target_queries[
                None, :, :
            ].expand(
                B,
                -1,
                -1,
            )
        )

        attended, _ = (
            self.cross_attention(
                query=queries,
                key=slot_tokens,
                value=slot_tokens,
                key_padding_mask=(
                    slot_mask <= 0
                ),
                need_weights=False,
            )
        )

        h = self.norm1(
            queries
            + self.dropout(
                attended
            )
        )

        h = self.norm2(
            h
            + self.dropout(
                self.ffn(h)
            )
        )

        logits = (
            h
            * self.target_head_weight[
                None, :, :
            ]
        ).sum(dim=-1)

        return (
            logits
            + self.target_head_bias[
                None, :
            ]
        )

## 5. Resumable training helpers

The notebook searches all attached inputs for:

```text
phase4b2b1_refit_epoch_1.pt
...
phase4b2b1_refit_epoch_5.pt
```

and resumes from the highest valid epoch automatically.

In [ ]:
def compute_global_weight_mean(
    uids
):
    d = pseudo_indexed.loc[
        list(uids),
        WEIGHT_COLS,
    ]

    mean_w = (
        d.to_numpy(
            np.float32
        ).mean(
            axis=0
        )
    )

    mean_w = np.maximum(
        mean_w,
        1e-6,
    )

    return torch.tensor(
        mean_w,
        dtype=torch.float32,
        device=DEVICE,
    )


def macro_weighted_bce_unbiased(
    logits,
    targets,
    weights,
    global_mean_weight,
):
    raw = (
        F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )
    )

    per_target = (
        (raw * weights).mean(
            dim=0
        )
        / global_mean_weight
    )

    return per_target.mean()


def make_optimizer(model):
    backbone_params = list(
        model.backbone.parameters()
    )

    backbone_ids = {
        id(p)
        for p in backbone_params
    }

    head_params = [
        p
        for p in model.parameters()
        if id(p) not in backbone_ids
    ]

    return torch.optim.AdamW(
        [
            {
                "params":
                    backbone_params,

                "lr":
                    BACKBONE_LR,
            },
            {
                "params":
                    head_params,

                "lr":
                    HEAD_LR,
            },
        ],
        weight_decay=
            WEIGHT_DECAY,
    )


def train_epoch(
    model,
    loader,
    optimizer,
    scaler,
    global_mean_weight,
):
    model.train()

    running = 0.0
    n_batches = 0

    for batch in tqdm(
        loader,
        leave=False,
        desc="train",
    ):
        images = batch[
            "images"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        slice_mask = batch[
            "slice_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        slot_mask = batch[
            "slot_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch[
            "targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        weights = batch[
            "weights"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):
            logits = model(
                images,
                slice_mask,
                slot_mask,
            )

            loss = (
                macro_weighted_bce_unbiased(
                    logits,
                    targets,
                    weights,
                    global_mean_weight,
                )
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP,
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        running += float(
            loss.detach().cpu()
        )

        n_batches += 1

    return (
        running
        / max(
            n_batches,
            1,
        )
    )


def discover_latest_refit_checkpoint():
    candidates = []

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        try:
            candidates.extend(
                root.rglob(
                    "phase4b2b1_refit_epoch_*.pt"
                )
            )
        except Exception:
            pass

    parsed = []

    for p in candidates:
        m = re.search(
            r"refit_epoch_(\d+)\.pt$",
            p.name,
        )

        if m:
            parsed.append(
                (
                    int(
                        m.group(1)
                    ),
                    p,
                )
            )

    parsed = [
        x for x in parsed
        if 0 < x[0] <= BEST_EPOCH
    ]

    if not parsed:
        return None, 0

    parsed.sort(
        key=lambda x: (
            x[0],
            str(x[1]),
        ),
        reverse=True,
    )

    return (
        parsed[0][1],
        parsed[0][0],
    )

## 6. Final refit — exactly 5 epochs

In [ ]:
train_loader = make_loader(
    all_pseudo_uids,
    training=True,
    use_gold_labels=False,
)

global_mean_weight = (
    compute_global_weight_mean(
        all_pseudo_uids
    )
)

resume_path, completed_epoch = (
    discover_latest_refit_checkpoint()
)

if resume_path is None:
    print(
        "No refit checkpoint found. "
        "Starting clean final refit from ImageNet."
    )

    final_model = (
        SixSlotTargetAttention(
            dropout=DROPOUT
        ).to(DEVICE)
    )

    optimizer = make_optimizer(
        final_model
    )

    scaler = (
        torch.cuda.amp.GradScaler(
            enabled=(
                DEVICE.type
                == "cuda"
            )
        )
    )

    history_rows = []

else:
    print(
        "Resuming from:",
        resume_path,
    )

    resume_payload = torch.load(
        resume_path,
        map_location="cpu",
        weights_only=False,
    )

    final_model = (
        SixSlotTargetAttention(
            dropout=DROPOUT
        ).to(DEVICE)
    )

    final_model.load_state_dict(
        resume_payload[
            "model_state_dict"
        ],
        strict=True,
    )

    optimizer = make_optimizer(
        final_model
    )

    optimizer.load_state_dict(
        resume_payload[
            "optimizer_state_dict"
        ]
    )

    scaler = (
        torch.cuda.amp.GradScaler(
            enabled=(
                DEVICE.type
                == "cuda"
            )
        )
    )

    if (
        "scaler_state_dict"
        in resume_payload
        and resume_payload[
            "scaler_state_dict"
        ]
    ):
        scaler.load_state_dict(
            resume_payload[
                "scaler_state_dict"
            ]
        )

    history_rows = list(
        resume_payload.get(
            "history",
            [],
        )
    )

    print(
        "Completed epoch:",
        completed_epoch,
    )


RUN_START = time.time()

for epoch in range(
    completed_epoch + 1,
    BEST_EPOCH + 1,
):
    t0 = time.time()

    loss = train_epoch(
        final_model,
        train_loader,
        optimizer,
        scaler,
        global_mean_weight,
    )

    seconds = (
        time.time()
        - t0
    )

    history_rows.append(
        {
            "epoch": epoch,
            "train_macro_weighted_bce": loss,
            "seconds": seconds,
        }
    )

    epoch_ckpt = (
        OUT_DIR
        / f"phase4b2b1_refit_epoch_{epoch}.pt"
    )

    torch.save(
        {
            "experiment_version":
                EXPERIMENT_VERSION,

            "epoch":
                epoch,

            "target_epoch":
                BEST_EPOCH,

            "model_state_dict":
                final_model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "history":
                history_rows,

            "selection_best_epoch":
                BEST_EPOCH,

            "selection_best_pseudo_val_loss":
                selection_checkpoint[
                    "best_pseudo_val_loss"
                ],

            "supervision":
                "PSEUDO_LABELS_v1_1",
        },
        epoch_ckpt,
    )

    print(
        f"refit epoch={epoch}/{BEST_EPOCH} "
        f"loss={loss:.5f} "
        f"time={seconds/60:.1f}m"
    )

    print(
        "Saved resumable checkpoint:",
        epoch_ckpt.name,
    )


final_history = pd.DataFrame(
    history_rows
)

display(final_history)

assert (
    int(
        final_history[
            "epoch"
        ].max()
    )
    == BEST_EPOCH
)

print(
    "\n✅ Five-epoch final refit complete"
)

## 6B. Save the final model immediately

This happens **before gold evaluation**.

So even if Kaggle times out later while computing metrics or making visualizations, the fully refit model checkpoint already exists in `/kaggle/working`.

In [ ]:
EARLY_FINAL_CKPT = (
    OUT_DIR
    / "phase4b2b1_final_sixslot_targetattn_resnet18_v1.pt"
)

torch.save(
    {
        "experiment_version":
            EXPERIMENT_VERSION,

        "model_state_dict":
            final_model.state_dict(),

        "model_config": {
            "targets":
                TARGETS,

            "slots":
                SLOTS,

            "crop_mm":
                CROP_MM,

            "image_size":
                IMAGE_SIZE,

            "train_anchors":
                N_ANCHORS,

            "inference_anchors":
                N_ANCHORS,

            "slice_band":
                list(SLICE_BAND),

            "dropout":
                DROPOUT,

            "architecture":
                "SixSlotTargetAttention_ResNet18_2p5D",
        },

        "selection": {
            "best_epoch":
                BEST_EPOCH,

            "best_pseudo_val_macro_weighted_bce":
                float(
                    selection_checkpoint[
                        "best_pseudo_val_loss"
                    ]
                ),

            "gold_used_for_selection":
                False,
        },

        "supervision": {
            "pseudolabel_freeze":
                "PSEUDO_LABELS_v1_1",

            "n_pseudo_train":
                4341,

            "n_gold_locked":
                58,

            "n_gold_group_neighbors_locked":
                6,
        },
    },
    EARLY_FINAL_CKPT,
)

EARLY_HISTORY_OUT = (
    OUT_DIR
    / "rsna_phase4b2b1_final_refit_history_v1.csv"
)

final_history.to_csv(
    EARLY_HISTORY_OUT,
    index=False,
)

print(
    "✅ Final model saved before evaluation:",
    EARLY_FINAL_CKPT,
)

## 7. Locked 58-study gold evaluation

In [ ]:
@torch.inference_mode()
def predict_loader(
    model,
    loader,
):
    model.eval()

    uid_list = []
    logits_list = []
    targets_list = []

    for batch in tqdm(
        loader,
        desc="gold inference",
    ):
        images = batch[
            "images"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        slice_mask = batch[
            "slice_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        slot_mask = batch[
            "slot_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):
            logits = model(
                images,
                slice_mask,
                slot_mask,
            )

        uid_list.extend(
            list(
                batch[
                    "StudyInstanceUID"
                ]
            )
        )

        logits_list.append(
            logits.float().cpu().numpy()
        )

        targets_list.append(
            batch[
                "targets"
            ].numpy()
        )

    return (
        uid_list,
        np.concatenate(
            logits_list,
            axis=0,
        ),
        np.concatenate(
            targets_list,
            axis=0,
        ),
    )


gold_loader = make_loader(
    gold_uids,
    training=False,
    use_gold_labels=True,
    batch_size=1,
)

gold_uid_order, gold_logits, gold_y = (
    predict_loader(
        final_model,
        gold_loader,
    )
)

gold_probs = (
    1.0
    / (
        1.0
        + np.exp(
            -gold_logits
        )
    )
)

rows = []

for j, target in enumerate(TARGETS):
    y = gold_y[:, j]
    p = gold_probs[:, j]

    auc = (
        roc_auc_score(
            y,
            p,
        )
        if len(
            np.unique(y)
        ) == 2
        else np.nan
    )

    ap = (
        average_precision_score(
            y,
            p,
        )
        if len(
            np.unique(y)
        ) == 2
        else np.nan
    )

    rows.append(
        {
            "target":
                target,

            "roc_auc":
                auc,

            "average_precision":
                ap,

            "n_positive":
                int(
                    y.sum()
                ),

            "n":
                len(y),
        }
    )


gold_metrics = pd.DataFrame(
    rows
)

macro_auc = float(
    gold_metrics[
        "roc_auc"
    ].mean()
)

macro_ap = float(
    gold_metrics[
        "average_precision"
    ].mean()
)

print(
    "Gold macro ROC-AUC:",
    macro_auc,
)

print(
    "Gold macro AP:",
    macro_ap,
)

display(
    gold_metrics
)

## 8. Save final submission-ready checkpoint and artifacts

In [ ]:
FINAL_CKPT = (
    OUT_DIR
    / "phase4b2b1_final_sixslot_targetattn_resnet18_v1.pt"
)

torch.save(
    {
        "experiment_version":
            EXPERIMENT_VERSION,

        "model_state_dict":
            final_model.state_dict(),

        "model_config": {
            "targets":
                TARGETS,

            "slots":
                SLOTS,

            "crop_mm":
                CROP_MM,

            "image_size":
                IMAGE_SIZE,

            "train_anchors":
                N_ANCHORS,

            "inference_anchors":
                N_ANCHORS,

            "slice_band":
                list(
                    SLICE_BAND
                ),

            "dropout":
                DROPOUT,

            "architecture":
                "SixSlotTargetAttention_ResNet18_2p5D",
        },

        "selection": {
            "best_epoch":
                BEST_EPOCH,

            "best_pseudo_val_macro_weighted_bce":
                float(
                    selection_checkpoint[
                        "best_pseudo_val_loss"
                    ]
                ),

            "gold_used_for_selection":
                False,
        },

        "supervision": {
            "pseudolabel_freeze":
                "PSEUDO_LABELS_v1_1",

            "n_pseudo_train":
                4341,

            "n_gold_locked":
                58,

            "n_gold_group_neighbors_locked":
                6,
        },
    },
    FINAL_CKPT,
)


FINAL_HISTORY_OUT = (
    OUT_DIR
    / "rsna_phase4b2b1_final_refit_history_v1.csv"
)

GOLD_METRICS_OUT = (
    OUT_DIR
    / "rsna_phase4b2b1_gold_metrics_v1.csv"
)

GOLD_PREDS_OUT = (
    OUT_DIR
    / "rsna_phase4b2b1_gold_predictions_v1.csv"
)

META_OUT = (
    OUT_DIR
    / "PHASE4B2B1_SIXSLOT_TARGETATTN_RESNET18_v1.json"
)


final_history.to_csv(
    FINAL_HISTORY_OUT,
    index=False,
)

gold_metrics.to_csv(
    GOLD_METRICS_OUT,
    index=False,
)


gold_predictions = pd.DataFrame({
    "StudyInstanceUID":
        gold_uid_order,
})

for j, target in enumerate(TARGETS):
    gold_predictions[
        f"gold_{TARGET_SLUG[target]}"
    ] = gold_y[:, j]

    gold_predictions[
        f"prob_{TARGET_SLUG[target]}"
    ] = gold_probs[:, j]

gold_predictions.to_csv(
    GOLD_PREDS_OUT,
    index=False,
)


meta = {
    "experiment_version":
        EXPERIMENT_VERSION,

    "phase4b2b1_complete":
        True,

    "selection": {
        "best_epoch":
            BEST_EPOCH,

        "best_pseudo_val_macro_weighted_bce":
            float(
                selection_checkpoint[
                    "best_pseudo_val_loss"
                ]
            ),

        "selection_checkpoint_source":
            str(
                USER_SELECTION_PATH
            ),
    },

    "final_refit": {
        "n_studies":
            4341,

        "epochs":
            BEST_EPOCH,

        "initialization":
            "ImageNet ResNet18",

        "resumable_epoch_checkpoints":
            True,
    },

    "gold_results": {
        "n":
            58,

        "macro_roc_auc":
            macro_auc,

        "macro_average_precision":
            macro_ap,
    },

    "final_checkpoint":
        FINAL_CKPT.name,
}


with open(
    META_OUT,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        meta,
        f,
        indent=2,
    )


print("Saved:")

for p in [
    FINAL_CKPT,
    FINAL_HISTORY_OUT,
    GOLD_METRICS_OUT,
    GOLD_PREDS_OUT,
    META_OUT,
]:
    print(" -", p.name)

print(
    "\n✅ PHASE 4B-2B1 COMPLETE"
)

# Send these back

```text
PHASE4B2B1_SIXSLOT_TARGETATTN_RESNET18_v1.json
rsna_phase4b2b1_final_refit_history_v1.csv
rsna_phase4b2b1_gold_metrics_v1.csv
rsna_phase4b2b1_gold_predictions_v1.csv
executed notebook
```

Keep:

```text
phase4b2b1_final_sixslot_targetattn_resnet18_v1.pt
```

That is the **actual final model checkpoint** we will package for Submission #1.

# 9. Results visualization

These plots are descriptive.

We do **not** use them to change the already-frozen epoch count or supervision.

In [ ]:
# ------------------------------------------------------------
# 9A. Final-refit training curve
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(8, 4)
)

ax.plot(
    final_history["epoch"],
    final_history["train_macro_weighted_bce"],
    marker="o",
)

ax.set_xlabel(
    "Final-refit epoch"
)

ax.set_ylabel(
    "Macro weighted BCE"
)

ax.set_title(
    "Phase 4B-2B1 final-refit training loss"
)

ax.grid(
    True,
    alpha=0.25,
)

plt.tight_layout()

TRAIN_CURVE_FIG = (
    OUT_DIR
    / "phase4b2b1_visual_final_refit_loss.png"
)

fig.savefig(
    TRAIN_CURVE_FIG,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    TRAIN_CURVE_FIG,
)

In [ ]:
# ------------------------------------------------------------
# 9B. Per-target gold ROC-AUC
# ------------------------------------------------------------
auc_plot = (
    gold_metrics[
        [
            "target",
            "roc_auc",
        ]
    ]
    .sort_values(
        "roc_auc",
        ascending=True,
    )
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.barh(
    auc_plot["target"],
    auc_plot["roc_auc"],
)

ax.axvline(
    macro_auc,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"Macro AUC = "
        f"{macro_auc:.3f}"
    ),
)

ax.set_xlim(
    0,
    1,
)

ax.set_xlabel(
    "ROC-AUC"
)

ax.set_title(
    "Locked gold performance by target"
)

ax.legend()

plt.tight_layout()

AUC_FIG = (
    OUT_DIR
    / "phase4b2b1_visual_gold_auc_by_target.png"
)

fig.savefig(
    AUC_FIG,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    AUC_FIG,
)

In [ ]:
# ------------------------------------------------------------
# 9C. Per-target average precision
# ------------------------------------------------------------
ap_plot = (
    gold_metrics[
        [
            "target",
            "average_precision",
        ]
    ]
    .sort_values(
        "average_precision",
        ascending=True,
    )
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.barh(
    ap_plot["target"],
    ap_plot["average_precision"],
)

ax.axvline(
    macro_ap,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"Macro AP = "
        f"{macro_ap:.3f}"
    ),
)

ax.set_xlim(
    0,
    1,
)

ax.set_xlabel(
    "Average precision"
)

ax.set_title(
    "Locked gold average precision by target"
)

ax.legend()

plt.tight_layout()

AP_FIG = (
    OUT_DIR
    / "phase4b2b1_visual_gold_ap_by_target.png"
)

fig.savefig(
    AP_FIG,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    AP_FIG,
)

# 10. MRI case visualization

We now choose **one gold study deterministically**:

1. prefer the gold study with the largest number of available MRI slots;
2. break ties by `StudyInstanceUID`.

This is intentionally not:

- the best case,
- the worst case,
- the highest-confidence case.

That avoids cherry-picking for the visualization.

The study was never used in model training or epoch selection.

In [ ]:
gold_slot_counts = (
    slot_plan.loc[
        slot_plan[
            "StudyInstanceUID"
        ].isin(
            gold_uids
        )
    ]
    .groupby(
        "StudyInstanceUID"
    )["slot"]
    .nunique()
)

gold_case_table = pd.DataFrame({
    "StudyInstanceUID":
        gold_uids,
})

gold_case_table[
    "n_slots"
] = (
    gold_case_table[
        "StudyInstanceUID"
    ]
    .map(
        gold_slot_counts
    )
    .fillna(0)
)

gold_case_table = (
    gold_case_table
    .sort_values(
        [
            "n_slots",
            "StudyInstanceUID",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

CASE_UID = str(
    gold_case_table.iloc[
        0
    ][
        "StudyInstanceUID"
    ]
)

print(
    "Selected deterministic gold case:",
    CASE_UID,
)

print(
    "Available slots:",
    sorted(
        slot_plan.loc[
            slot_plan[
                "StudyInstanceUID"
            ].eq(
                CASE_UID
            ),
            "slot",
        ].tolist()
    ),
)

In [ ]:
case_images, case_slice_mask, case_slot_mask = (
    build_one_study(
        CASE_UID
    )
)

fig, axes = plt.subplots(
    len(SLOTS),
    N_ANCHORS,
    figsize=(16, 11),
)

for s, slot in enumerate(
    SLOTS
):
    for k in range(
        N_ANCHORS
    ):
        ax = axes[s, k]

        if (
            case_slot_mask[s] > 0
            and case_slice_mask[
                s,
                k,
            ] > 0
        ):
            # center channel of [previous, center, next]
            ax.imshow(
                case_images[
                    s,
                    k,
                    1,
                ],
                cmap="gray",
                vmin=0,
                vmax=1,
            )

        else:
            ax.text(
                0.5,
                0.5,
                "missing",
                ha="center",
                va="center",
                fontsize=8,
            )

        ax.axis(
            "off"
        )

        if s == 0:
            ax.set_title(
                f"A{k+1}"
            )

        if k == 0:
            ax.text(
                -0.10,
                0.5,
                slot,
                transform=ax.transAxes,
                ha="right",
                va="center",
                fontsize=9,
            )

fig.suptitle(
    (
        f"Gold study {CASE_UID}: "
        "130-mm cropped MRI anchors"
    ),
    y=0.995,
)

plt.tight_layout()

MRI_MONTAGE_FIG = (
    OUT_DIR
    / "phase4b2b1_visual_gold_case_mri_montage.png"
)

fig.savefig(
    MRI_MONTAGE_FIG,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    MRI_MONTAGE_FIG,
)

# 11. Explain what the model attended to

There are two learned attention stages:

```text
8 anchors inside a series
        ↓
within-slot attention
        ↓
one token per MRI slot

6 possible MRI slots
        ↓
12 target-specific queries
        ↓
one evidence mixture per target
```

Attention is useful for debugging and intuition, but it is **not a clinical causal explanation**.

In [ ]:
@torch.inference_mode()
def explain_final_model(
    model,
    images,
    slice_mask,
    slot_mask,
):
    model.eval()

    B, S, K, C, H, W = (
        images.shape
    )

    flat = images.reshape(
        B * S * K,
        C,
        H,
        W,
    )

    features = (
        model.backbone(
            flat
        )
    )

    features = (
        features.reshape(
            B,
            S,
            K,
            -1,
        )
    )

    scores = (
        model.slice_score(
            features
        ).squeeze(
            -1
        )
    )

    m = (
        slice_mask.float()
    )

    scores = (
        scores
        - scores.amax(
            dim=-1,
            keepdim=True,
        )
    )

    exp_scores = (
        torch.exp(
            scores
        )
        * m
    )

    slice_attention = (
        exp_scores
        / (
            exp_scores.sum(
                dim=-1,
                keepdim=True,
            )
            + 1e-8
        )
    )

    slot_tokens = (
        features
        * slice_attention.unsqueeze(
            -1
        )
    ).sum(
        dim=2
    )

    slot_ids = torch.arange(
        S,
        device=images.device,
    )

    slot_tokens = (
        slot_tokens
        + model.slot_embedding(
            slot_ids
        )[None, :, :]
    )

    queries = (
        model.target_queries[
            None, :, :
        ].expand(
            B,
            -1,
            -1,
        )
    )

    attended, target_slot_attention = (
        model.cross_attention(
            query=queries,
            key=slot_tokens,
            value=slot_tokens,
            key_padding_mask=(
                slot_mask <= 0
            ),
            need_weights=True,
            average_attn_weights=True,
        )
    )

    h = (
        model.norm1(
            queries
            + model.dropout(
                attended
            )
        )
    )

    h = (
        model.norm2(
            h
            + model.dropout(
                model.ffn(
                    h
                )
            )
        )
    )

    logits = (
        h
        * model.target_head_weight[
            None, :, :
        ]
    ).sum(
        dim=-1
    )

    logits = (
        logits
        + model.target_head_bias[
            None, :
        ]
    )

    return (
        logits,
        slice_attention,
        target_slot_attention,
    )


case_images_t = (
    torch.from_numpy(
        case_images[
            None
        ]
    )
    .to(
        DEVICE
    )
)

case_slice_mask_t = (
    torch.from_numpy(
        case_slice_mask[
            None
        ]
    )
    .to(
        DEVICE
    )
)

case_slot_mask_t = (
    torch.from_numpy(
        case_slot_mask[
            None
        ]
    )
    .to(
        DEVICE
    )
)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
    enabled=(
        DEVICE.type
        == "cuda"
    ),
):
    (
        case_logits_t,
        case_slice_attn_t,
        case_target_slot_attn_t,
    ) = explain_final_model(
        final_model,
        case_images_t,
        case_slice_mask_t,
        case_slot_mask_t,
    )

case_probs = (
    torch.sigmoid(
        case_logits_t
    )
    .float()
    .cpu()
    .numpy()[0]
)

case_slice_attention = (
    case_slice_attn_t
    .float()
    .cpu()
    .numpy()[0]
)

case_target_slot_attention = (
    case_target_slot_attn_t
    .float()
    .cpu()
    .numpy()[0]
)

print(
    "Model probabilities:",
)

display(
    pd.DataFrame({
        "target":
            TARGETS,

        "MRI_probability":
            case_probs,
    })
)

In [ ]:
# ------------------------------------------------------------
# 11A. Within-slot anchor attention
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(11, 5)
)

im = ax.imshow(
    case_slice_attention,
    aspect="auto",
)

ax.set_yticks(
    np.arange(
        len(SLOTS)
    )
)

ax.set_yticklabels(
    SLOTS
)

ax.set_xticks(
    np.arange(
        N_ANCHORS
    )
)

ax.set_xticklabels([
    f"A{i+1}"
    for i in range(
        N_ANCHORS
    )
])

ax.set_xlabel(
    "2.5D anchor"
)

ax.set_title(
    (
        "Within-series attention: "
        "which MRI anchors received more weight?"
    )
)

fig.colorbar(
    im,
    ax=ax,
    label="attention weight",
)

plt.tight_layout()

SLICE_ATTN_FIG = (
    OUT_DIR
    / "phase4b2b1_visual_gold_case_slice_attention.png"
)

fig.savefig(
    SLICE_ATTN_FIG,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    SLICE_ATTN_FIG,
)

In [ ]:
# ------------------------------------------------------------
# 11B. Target-to-MRI-slot attention
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8)
)

im = ax.imshow(
    case_target_slot_attention,
    aspect="auto",
)

ax.set_yticks(
    np.arange(
        len(TARGETS)
    )
)

ax.set_yticklabels(
    TARGETS
)

ax.set_xticks(
    np.arange(
        len(SLOTS)
    )
)

ax.set_xticklabels(
    SLOTS,
    rotation=35,
    ha="right",
)

ax.set_title(
    (
        "Target-specific MRI slot attention\n"
        "(post-hoc diagnostic, not a causal explanation)"
    )
)

fig.colorbar(
    im,
    ax=ax,
    label="attention weight",
)

plt.tight_layout()

TARGET_ATTN_FIG = (
    OUT_DIR
    / "phase4b2b1_visual_gold_case_target_slot_attention.png"
)

fig.savefig(
    TARGET_ATTN_FIG,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    TARGET_ATTN_FIG,
)

# 12. MRI prediction vs official gold label vs frozen report-derived target

For this one locked study we compare three things:

```text
Official gold label
    = expert ground truth

MRI probability
    = final Phase 4B-2B1 image model

Frozen report-derived pseudo target
    = Rule v0.2 + continuous E5 teacher used in our supervision pipeline
```

The report-derived target was created before this model and remains frozen.

This comparison is for understanding only.

In [ ]:
case_gold = (
    gold_label_table.loc[
        CASE_UID,
        TARGETS,
    ]
    .to_numpy(
        np.float32
    )
)

if CASE_UID in pseudo_indexed.index:
    case_pseudo = (
        pseudo_indexed.loc[
            CASE_UID,
            PSEUDO_COLS,
        ]
        .to_numpy(
            np.float32
        )
    )

else:
    case_pseudo = (
        np.full(
            len(TARGETS),
            np.nan,
            dtype=np.float32,
        )
    )


case_compare = pd.DataFrame({
    "target":
        TARGETS,

    "gold_label":
        case_gold,

    "MRI_probability":
        case_probs,

    "report_teacher_target":
        case_pseudo,
})

display(
    case_compare
)


x = np.arange(
    len(TARGETS)
)

width = 0.25

fig, ax = plt.subplots(
    figsize=(15, 6)
)

ax.bar(
    x - width,
    case_gold,
    width,
    label="Official gold label",
)

ax.bar(
    x,
    case_probs,
    width,
    label="MRI model probability",
)

if np.isfinite(
    case_pseudo
).any():
    ax.bar(
        x + width,
        case_pseudo,
        width,
        label="Frozen report-derived target",
    )

ax.set_xticks(
    x
)

ax.set_xticklabels(
    TARGETS,
    rotation=45,
    ha="right",
)

ax.set_ylim(
    0,
    1.05,
)

ax.set_ylabel(
    "Label / probability"
)

ax.set_title(
    (
        f"Gold study {CASE_UID}: "
        "MRI vs report teacher vs official label"
    )
)

ax.legend()

plt.tight_layout()

CASE_COMPARE_FIG = (
    OUT_DIR
    / "phase4b2b1_visual_gold_case_mri_report_gold_compare.png"
)

fig.savefig(
    CASE_COMPARE_FIG,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    CASE_COMPARE_FIG,
)

# 13. Find and display the clinical report text

The raw report may live in a different attached dataset depending on how your Phase 4A files were saved.

This cell searches attached CSV/Parquet files for:

- `StudyInstanceUID`
- a report-like text column such as `Report`, `report_text`, or similar.

If the raw report is not attached, the notebook does **not** invent one.  
The report-teacher probabilities above are still valid because they come from the frozen pseudo-label table.

In [ ]:
REPORT_TEXT_ALIASES = [
    "Report",
    "report",
    "report_text",
    "ReportText",
    "clinical_report",
    "ClinicalReport",
    "report_clean",
    "text",
]


def find_report_source(
    case_uid,
):
    roots = [
        Path(
            "/kaggle/input"
        ),
        Path(
            "/kaggle/working"
        ),
    ]

    # Prefer files whose names mention report.
    candidate_paths = []

    for root in roots:
        if not root.exists():
            continue

        for pattern in [
            "*report*.csv",
            "*report*.parquet",
            "*reports*.csv",
            "*reports*.parquet",
            "train.csv",
        ]:
            try:
                candidate_paths.extend(
                    root.rglob(
                        pattern
                    )
                )
            except Exception:
                pass

    # Deduplicate.
    seen = set()
    candidates = []

    for p in candidate_paths:
        key = str(p)

        if key not in seen:
            seen.add(key)
            candidates.append(
                p
            )

    print(
        "Report-table candidates:",
        len(candidates),
    )

    for p in candidates:
        try:
            if (
                p.suffix.lower()
                == ".csv"
            ):
                head = pd.read_csv(
                    p,
                    nrows=5,
                )

            elif (
                p.suffix.lower()
                == ".parquet"
            ):
                head = pd.read_parquet(
                    p
                ).head(
                    5
                )

            else:
                continue

        except Exception:
            continue

        if (
            "StudyInstanceUID"
            not in head.columns
        ):
            continue

        text_col = None

        for alias in REPORT_TEXT_ALIASES:
            if alias in head.columns:
                text_col = alias
                break

        if text_col is None:
            report_like = [
                c
                for c in head.columns
                if "report"
                in str(c).lower()
            ]

            if report_like:
                text_col = (
                    report_like[0]
                )

        if text_col is None:
            continue

        try:
            if (
                p.suffix.lower()
                == ".csv"
            ):
                d = pd.read_csv(
                    p,
                    usecols=[
                        "StudyInstanceUID",
                        text_col,
                    ],
                )

            else:
                d = pd.read_parquet(
                    p,
                    columns=[
                        "StudyInstanceUID",
                        text_col,
                    ],
                )

        except Exception:
            continue

        d[
            "StudyInstanceUID"
        ] = (
            d[
                "StudyInstanceUID"
            ].astype(
                str
            )
        )

        hit = d.loc[
            d[
                "StudyInstanceUID"
            ].eq(
                str(
                    case_uid
                )
            )
        ]

        if len(
            hit
        ):
            text = hit.iloc[
                0
            ][
                text_col
            ]

            if (
                pd.notna(
                    text
                )
                and str(
                    text
                ).strip()
            ):
                return (
                    p,
                    text_col,
                    str(
                        text
                    ),
                )

    return (
        None,
        None,
        None,
    )


(
    report_source,
    report_column,
    report_text,
) = find_report_source(
    CASE_UID
)


if report_text is None:
    print(
        "⚠️ Raw clinical report text was not found "
        "in the currently attached inputs."
    )

    print(
        "This does not affect MRI training/evaluation."
    )

else:
    print(
        "Report source:",
        report_source,
    )

    print(
        "Report column:",
        report_column,
    )

    print()
    print(
        "----- CLINICAL REPORT -----"
    )
    print(
        report_text[
            :5000
        ]
    )
    print(
        "----- END REPORT -----"
    )

# 14. How to read this case review

The three channels answer different questions:

### MRI montage
Shows the actual 130-mm cropped anatomical evidence fed to the image network.

### Attention heatmaps
Show where the learned network put more numerical weight:

```text
slice attention:
which anchors inside a series?

target-slot attention:
which MRI sequence types for each target?
```

### Clinical report + report teacher
Shows the textual evidence from the radiology workflow and the frozen soft labels we derived from it.

A disagreement such as:

```text
MRI probability high
report teacher low
gold positive
```

can indicate the image model is rescuing a report-teacher miss.

The opposite:

```text
MRI probability low
report teacher high
gold positive
```

can indicate that the report teacher contains useful complementary information.

Those disagreements are exactly what we will later use to decide whether an **MRI ensemble / multimodal training strategy** is worth pursuing.

## But for the competition

The final test-time model must only use information available in the hidden test set.  
This notebook therefore keeps report inspection as a **training/evaluation diagnostic**, not a dependency of the MRI inference model.

In [ ]:
# ------------------------------------------------------------
# Save a compact case-review CSV
# ------------------------------------------------------------
CASE_REVIEW_OUT = (
    OUT_DIR
    / "phase4b2b1_gold_case_review.csv"
)

case_compare[
    "StudyInstanceUID"
] = (
    CASE_UID
)

case_compare[
    "report_text_found"
] = (
    report_text
    is not None
)

case_compare.to_csv(
    CASE_REVIEW_OUT,
    index=False,
)

print(
    "Saved:",
    CASE_REVIEW_OUT,
)

print()
print(
    "✅ TRAINING + GOLD EVALUATION + VISUAL QA COMPLETE"
)

# Send these results back to me

The most important files are:

```text
phase4b2b1_final_sixslot_targetattn_resnet18_v1.pt

PHASE4B2B1_SIXSLOT_TARGETATTN_RESNET18_v1.json

rsna_phase4b2b1_final_refit_history_v1.csv
rsna_phase4b2b1_gold_metrics_v1.csv
rsna_phase4b2b1_gold_predictions_v1.csv

phase4b2b1_gold_case_review.csv
```

And if possible send the executed notebook or screenshots of:

```text
phase4b2b1_visual_final_refit_loss.png
phase4b2b1_visual_gold_auc_by_target.png
phase4b2b1_visual_gold_ap_by_target.png

phase4b2b1_visual_gold_case_mri_montage.png
phase4b2b1_visual_gold_case_slice_attention.png
phase4b2b1_visual_gold_case_target_slot_attention.png
phase4b2b1_visual_gold_case_mri_report_gold_compare.png
```

Then we will decide whether this model should become **Submission #1** by itself or be blended with our earlier MRI baseline.